# 03 · Evaluate (GPU)

Scores zero-shot, every DAPT adapter, and each task row on the TEST splits → `results/`. Verdict Macro-F1 + ROUGE always; `use_judge=True` adds the CLEV LLM-judge. Judge models come from `FEDDAPT_JUDGE_MODELS` — API (`claude-…`) or local (`ollama:…`), and must differ from the Mistral-7B base.

In [ ]:
# --- setup: run once per Colab session (run me FIRST) ---
import os, sys
# 1) get the code
if not os.path.exists('/content/fedapt/pyproject.toml') and not os.path.exists('pyproject.toml'):
    get_ipython().system('git clone https://github.com/dsuyu1/fedapt.git /content/fedapt')
if os.path.exists('/content/fedapt/pyproject.toml'):
    os.chdir('/content/fedapt')
# 2) install (no -q, so any error is visible). Colab has the GPU for [train]/[eval].
get_ipython().system('pip install -e ".[train,eval]"')
# 3) expose the src-layout package to THIS kernel (editable installs otherwise need a restart)
sys.path.insert(0, os.path.abspath('src'))
import fedapt; print('fedapt OK:', fedapt.__file__)
# 4) persist corpus/adapters/results to Drive so a dropped session resumes.
#    On Colab, Config auto-defaults FEDDAPT_ROOT to /content/drive/MyDrive/FedDAPT.
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
from fedapt.config import load_config
from fedapt import evaluate
cfg = load_config()

### Reference metrics only (no judge)

In [ ]:
evaluate.run_eval(cfg, use_judge=False)

### CLEV judge
Run this **after** the cell above frees the base model (student + judge both want the GPU). For a local judge, run the Ollama cell first and set `FEDDAPT_JUDGE_MODELS=ollama:gemma3:12b,ollama:llama3.1:8b`.

In [ ]:
# --- OPTIONAL: run a strong LOCAL teacher/judge on Colab's GPU via Ollama ---
# Open-weight models (Qwen/Gemma/Llama) run on the SAME GPU as training, so only
# do this during data-build (nb 00) or the judge pass (nb 03), never during training.
# T4: qwen3:14b / gemma3:12b.  A100: qwen3:32b / gemma3:27b.
get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')
import subprocess, time, os
subprocess.Popen(['ollama', 'serve']); time.sleep(5)
get_ipython().system('ollama pull qwen3:14b')
os.environ['FEDDAPT_OLLAMA_HOST'] = 'http://localhost:11434'

In [ ]:
# evaluate.run_eval(cfg, use_judge=True)   # validate the judge first: scripts/score_judge.py

Next → **04 Analysis**.